In [29]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig, set_seed, EarlyStoppingCallback
from peft import LoraConfig
from trl import SFTTrainer
from datasets import load_dataset
import os
import torch
from tqdm import tqdm
import re
import editdistance
from sklearn.model_selection import KFold
import numpy as np
import random
import torch
from peft import PeftModel
import json

# Setup Environments

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
seed = 42

random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # For multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
set_seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# Load Training Dataset

In [4]:
dataset = load_dataset("json", data_files="val_candidates_seed1_6_9_11_15_17_20_21_26_28_top1.json", split="train")
num_folds = 5
kf = KFold(n_splits=num_folds, shuffle=True, random_state=seed)

# LLM Configs

In [5]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Llama has no pad token by default, eos is fine for Llama because it uses end of turn to stop chat generation
tokenizer.padding_side = "right" # Fix for fp16 training
tokenizer.pad_token_id = tokenizer.eos_token_id

peft_config = LoraConfig(
    r=16,       
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [6]:
def format_chat_template(row):
    messages = [
        {
            "role": "system",
            # "content": "Your task is to perform automatic speech recognition. Below are multiple candidate transcriptions. Based on the transcription candidates, come up with a transcription that is most accurate, ensuring the transcription is contextually and grammatically correct. Focus on key differences in the candidates that change the meaning or correctness. Avoid selections with repetitive or nonsensical phrases. In cases of ambiguity, select the option that is most coherent and contextually sound. Respond with your refined transcription only, without any introductory text."
            # "content": "Your task is to perform automatic speech recognition. Below are a list of candidate transcriptions for a single spoken sentence. Identify the true transcription from the list. The candidates may contain phonetic errors (words that sound similar but are wrong). Use your knowledge of context, grammar, and common speech patterns to select the most likely original sentence. Choose EXACTLY one transcription from the list or a corrected version if all contain slight errors. Respond ONLY with the final text. Do not provide explanations, introductory text, or punctuation. Output lowercase only."
            "content": "Your task is to perform automatic speech recognition. Below are multiple candidate transcriptions predicted by different models. Choose the transcription that is most accurate, ensuring it is contextually and grammatically correct. Focus on key differences in the options that change the meaning or correctness. Avoid selections with repetitive or nonsensical phrases. In cases of ambiguity, select the option that is most coherent and contextually sound. Make sure to respond the complete chosen transcription exactly as it appears, word-for-word, no capitalization. Respond with the chosen transcription only, without any introductory text."
        },
        {
            "role": "user",
            "content": "\n".join([f"{cand}" for i, cand in enumerate(row["candidates"])])
        },
        {
            "role": "assistant",
            "content": row["ground_truth"]
        }
    ]
    row["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
    return row

# Training

In [7]:
print(f"Starting {num_folds}-Fold Cross-Validation on {len(dataset)} samples...")

for fold_idx, (train_indices, val_indices) in enumerate(kf.split(dataset)):
    
    print(f"\n--- Starting Fold {fold_idx+1}/{num_folds} ---")

    train_dataset = dataset.select(train_indices)
    val_dataset = dataset.select(val_indices)

    train_dataset = train_dataset.map(format_chat_template)
    val_dataset = val_dataset.map(format_chat_template)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        # quantization_config=bnb_config,
        device_map="cuda:0",
        dtype=torch.float16
    )
    model.config.use_cache = False # Don't use during training
    model.config.pretraining_tp = 1 # Set to 1

    training_args = TrainingArguments(
        output_dir="./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold"+str(fold_idx+1),
        num_train_epochs=10.0,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        # dataloader_drop_last=True,
        learning_rate=5e-5,
        weight_decay=0.1,
        fp16=True,  # Use mixed precision
        logging_dir="./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold"+str(fold_idx+1)+"/logs",
        logging_strategy="steps",
        logging_steps=18,
        save_strategy="steps",
        save_steps=18,   
        eval_strategy="steps",              
        eval_steps=18,                      
        per_device_eval_batch_size=32, 
        max_grad_norm=1.0,
        warmup_ratio=0.05,
        label_smoothing_factor=0.0,
        group_by_length=False, # Sorts samples by length for efficiency
        lr_scheduler_type="cosine",
        seed=seed,
        data_seed=seed,
        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        greater_is_better=False,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
        peft_config=peft_config,
        args=training_args,
    )

    trainer.train()

Starting 5-Fold Cross-Validation on 1426 samples...

--- Starting Fold 1/5 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.52it/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
18,2.674300,2.322484,1.807611,67608.000000,0.607376
36,1.837100,1.138523,1.217521,135464.000000,0.744257
54,0.588800,0.354003,0.361022,203078.000000,0.947194
72,0.317700,0.312775,0.309567,267807.000000,0.955197
90,0.301100,0.303964,0.294668,334956.000000,0.955867
108,0.283900,0.296234,0.284060,402522.000000,0.956072
126,0.297900,0.285300,0.296307,470486.000000,0.956001
144,0.283800,0.265062,0.290486,535614.000000,0.956091
162,0.238800,0.228088,0.201246,603304.000000,0.960319
180,0.229900,0.206571,0.249771,671360.000000,0.960376



--- Starting Fold 2/5 ---


Truncating eval dataset: 100%|██████████| 285/285 [00:00<00:00, 134705.50 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


...

Truncating eval dataset: 100%|██████████| 285/285 [00:00<00:00, 130942.78 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
18,2.669400,2.347725,1.818682,67467.000000,0.603820
36,1.830000,1.170868,1.237824,135134.000000,0.742152
54,0.604200,0.366648,0.367268,203056.000000,0.946054
72,0.336900,0.321887,0.324956,268512.000000,0.954610
90,0.292200,0.312335,0.300311,336528.000000,0.955307
108,0.306900,0.303419,0.305189,404208.000000,0.955639
126,0.293900,0.292011,0.292075,471311.000000,0.955591
144,0.268000,0.268849,0.293584,537024.000000,0.955663
162,0.245600,0.233643,0.206654,604207.000000,0.960203
180,0.213400,0.212906,0.229693,672051.000000,0.960081



--- Starting Fold 5/5 ---


Truncating eval dataset: 100%|██████████| 285/285 [00:00<00:00, 143882.60 examples/s]
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
18,2.703300,2.320764,1.793137,67211.000000,0.608387
36,1.817600,1.154369,1.223673,135196.000000,0.749835
54,0.613200,0.349096,0.358199,202755.000000,0.948197
72,0.335100,0.309712,0.311953,268032.000000,0.956441
90,0.309800,0.301163,0.293083,335302.000000,0.956803
108,0.297300,0.294252,0.291603,403149.000000,0.957025
126,0.285200,0.285210,0.290197,471237.000000,0.957080
144,0.293100,0.270282,0.286414,536064.000000,0.957207
162,0.235700,0.229564,0.247316,604222.000000,0.961447
180,0.234800,0.209125,0.240634,671624.000000,0.961572


# WER Evaluation

In [10]:
def remove_punctuation(sentence):

    sentence = re.sub(r'[^a-zA-Z\- \']', '', sentence)
    sentence = sentence.replace('- ', ' ').lower()
    sentence = sentence.replace('--', '').lower()
    sentence = sentence.replace(" '", "'").lower()
    sentence = sentence.strip()
    sentence = ' '.join([word for word in sentence.split() if word != ''])

    return sentence

def generate_predictions(model, tokenizer, val_dataset, batch_size=8):

    model.eval()
    
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    predictions = []
    ground_truths = []
    candidates_list = []
    
    device = next(model.parameters()).device

    formatted_prompts = []
    for sample in val_dataset:
        messages = [
            {
                "role": "system",
                "content": "Your task is to perform automatic speech recognition. Below are multiple candidate transcriptions predicted by different models. Choose the transcription that is most accurate, ensuring it is contextually and grammatically correct. Focus on key differences in the options that change the meaning or correctness. Avoid selections with repetitive or nonsensical phrases. In cases of ambiguity, select the option that is most coherent and contextually sound. Make sure to respond the complete chosen transcription exactly as it appears, word-for-word, no capitalization. Respond with the chosen transcription only, without any introductory text."
            },
            {
                "role": "user",
                "content": "\n".join(sample['candidates'])
            }
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # This adds <|start_header_id|>assistant<|end_header_id|>
        )
        formatted_prompts.append(prompt)
        ground_truths.append(sample['ground_truth'].strip())
        candidates_list.append(sample['candidates'])

    for i in range(0, len(formatted_prompts), batch_size):
        batch_prompts = formatted_prompts[i : i + batch_size]
        inputs = tokenizer(
            batch_prompts, 
            return_tensors="pt", 
            padding=True,       # Pads to the longest seq in this batch
            truncation=True, 
            max_length=2048
        ).to(device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,           # Limit response length
                use_cache=True,               # Enable cache for faster generation     
                pad_token_id=tokenizer.pad_token_id, 
                eos_token_id=tokenizer.eos_token_id,  
                # temperature=0.1,              # Low temperature for deterministic output
                # do_sample=True,               # Enable sampling for temperature
                do_sample=False,
                # top_p=0.9,                    # Nucleus sampling
                # repetition_penalty=1.1        # Prevent repetition
                repetition_penalty=1.0
            )
        generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        decoded_batch = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        clean_batch = [pred.strip().lower() for pred in decoded_batch]
        predictions.extend(clean_batch)
    
    return predictions, ground_truths, candidates_list

In [ ]:
steps = np.arange(144, 594+18, 18).tolist()

for step_idx, step in enumerate(steps):

    prediction_all = []
    ground_truths_all = []
    print(f"--- Starting Step {step} ---")

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(dataset)):
        
        # print(f"--- Starting Fold {fold_idx+1}/{num_folds} ---")

        val_dataset = dataset.select(val_indices)

        model_id = "meta-llama/Llama-3.2-3B-Instruct"
        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="cuda:0",
            torch_dtype=torch.float16
        )
        checkpoint_path = f"./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold{fold_idx+1}/checkpoint-{step}" 
        finetuned_model = PeftModel.from_pretrained(
            base_model,
            checkpoint_path,
            device_map="cuda:0"
        )
        finetuned_model.eval()

        # print(f"Loaded fine-tuned model from {checkpoint_path}")

        # tokenizer = trainer.tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)

        predictions_ft, ground_truths_ft, candidates_ft = generate_predictions(
            finetuned_model,  
            tokenizer,
            val_dataset,
            batch_size=16
        )

        prediction_all.extend(predictions_ft)
        ground_truths_all.extend(ground_truths_ft)

    results = []
    for i, (pred, gt) in enumerate(zip(prediction_all, ground_truths_all)):
        results.append({
            "id": i,
            "prediction": pred,
            "ground_truth": gt
        })
    output_file = f"./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/val_results/step_{step}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    total_true_length = 0
    total_edit_distance = 0

    for i in range(len(prediction_all)):
        true_sentence = remove_punctuation(ground_truths_all[i]).strip()
        pred_sentence = remove_punctuation(prediction_all[i]).strip()
        ed = editdistance.eval(true_sentence.split(), pred_sentence.split())
        total_edit_distance += ed
        total_true_length += len(true_sentence.split())

    print(f'Step {step} Aggregate Word Error Rate (WER): {100 * total_edit_distance / total_true_length:.2f}%')



--- Starting Fold 1/5 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.61it/s]


Loaded fine-tuned model from ./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold1/checkpoint-144

--- Starting Fold 2/5 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.59it/s]


Loaded fine-tuned model from ./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold2/checkpoint-144

--- Starting Fold 3/5 ---


...

Loaded fine-tuned model from ./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold3/checkpoint-594

--- Starting Fold 4/5 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.57it/s]


Loaded fine-tuned model from ./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold4/checkpoint-594

--- Starting Fold 5/5 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.58it/s]


Loaded fine-tuned model from ./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold5/checkpoint-594
Step 594 Aggregate Word Error Rate (WER): 3.16%


In [31]:
steps = np.arange(144, 594+18, 18).tolist()

for step_idx, step in enumerate(steps):

    prediction_all = []
    ground_truths_all = []
    print(f"--- Starting Step {step} ---")

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(dataset)):
        
        # print(f"--- Starting Fold {fold_idx+1}/{num_folds} ---")

        val_dataset = dataset.select(val_indices)

        model_id = "meta-llama/Llama-3.2-3B-Instruct"
        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="cuda:0",
            torch_dtype=torch.float16
        )
        checkpoint_path = f"./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/fold{fold_idx+1}/checkpoint-{step}" 
        finetuned_model = PeftModel.from_pretrained(
            base_model,
            checkpoint_path,
            device_map="cuda:0"
        )
        finetuned_model.eval()

        # print(f"Loaded fine-tuned model from {checkpoint_path}")

        # tokenizer = trainer.tokenizer
        tokenizer = AutoTokenizer.from_pretrained(model_id)

        predictions_ft, ground_truths_ft, candidates_ft = generate_predictions(
            finetuned_model,  
            tokenizer,
            val_dataset,
            batch_size=32
        )

        prediction_all.extend(predictions_ft)
        ground_truths_all.extend(ground_truths_ft)

    results = []
    for i, (pred, gt) in enumerate(zip(prediction_all, ground_truths_all)):
        results.append({
            "id": i,
            "prediction": pred,
            "ground_truth": gt
        })
    output_dir = "./Llama3.2-3B-Instruct_5fold_10epochs_rank16_10bestseeds/val_results"
    output_file = f"{output_dir}/step_{step}.json"
    os.makedirs(output_dir, exist_ok=True)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    total_true_length = 0
    total_edit_distance = 0

    for i in range(len(prediction_all)):
        true_sentence = remove_punctuation(ground_truths_all[i]).strip()
        pred_sentence = remove_punctuation(prediction_all[i]).strip()
        ed = editdistance.eval(true_sentence.split(), pred_sentence.split())
        total_edit_distance += ed
        total_true_length += len(true_sentence.split())

    print(f'Step {step} Aggregate Word Error Rate (WER): {100 * total_edit_distance / total_true_length:.2f}%')


--- Starting Step 144 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.58it/s]


Step 144 Aggregate Word Error Rate (WER): 3.32%
--- Starting Step 162 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.60it/s]


Step 162 Aggregate Word Error Rate (WER): 3.28%
--- Starting Step 180 ---


...

Step 252 Aggregate Word Error Rate (WER): 3.24%
--- Starting Step 270 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.58it/s]


Step 270 Aggregate Word Error Rate (WER): 3.21%
--- Starting Step 288 ---


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.62it/s]


KeyboardInterrupt: 